# Kaggle Titanic — 生存预测（手动数据）

1. 打开 [Titanic Data](https://www.kaggle.com/c/titanic/data)，登录后下载 **`train.csv`**、**`test.csv`**。
2. 将两个文件放到本 notebook 所在目录（或修改下方 **`DATA_DIR`** 为你的文件夹路径）。
3. 依次运行全部代码单元格，生成 **`submission.csv`**，在比赛页 **Submit Predictions** 上传提交。

依赖：`pip install pandas scikit-learn numpy`（一般环境已具备）。

---

## 工作流（整体在做什么）

下面是一条**竞赛里常见的表格分类流水线**：从原始 CSV 到可上传的提交文件，中间**绝不**用测试集的标签（测试集本来就没有 `Survived`），避免**数据泄漏**。

| 步骤 | 单元格 | 内容 |
|------|--------|------|
| 1 | 导入与路径 | 引入库，设定 `DATA_DIR`，检查 `train.csv` / `test.csv` 是否存在 |
| 2 | 特征函数 | 定义 `add_features`、`build_xy`、`build_x_test`（只定义函数，尚未读入大表） |
| 3 | 训练 | 读入 CSV → 构造 **X, y** → 搭建预处理 + 模型 **Pipeline** → **5 折交叉验证**估准确率 → 在**全训练集**上 **fit** |
| 4 | 提交 | 对 **test** 构造 **X_test** → **predict** → 拼 `PassengerId` → 保存 **submission.csv** |

**数据划分说明**：`train.csv` 用来学参数和做交叉验证；`test.csv` 只在最后预测一次，用来模拟 Kaggle 隐藏标签的测试集。

```mermaid
flowchart TD
    A[train.csv] --> B[特征工程 build_xy]
    B --> C[X 特征矩阵 y 标签]
    C --> D[预处理 ColumnTransformer]
    D --> E[分类器 GradientBoosting]
    E --> F[cross_val_score 评估]
    E --> G[fit 全量训练集]
    H[test.csv] --> I[build_x_test]
    I --> J[predict]
    G --> J
    J --> K[submission.csv]
```

**术语对照**：**X** = 模型输入特征表；**y** = 是否幸存 0/1；**Pipeline** = 先预处理再分类，避免手写步骤顺序出错；**StratifiedKFold** = 折内保持 `y` 的 0/1 比例接近原始分布。

In [3]:
# ---------- 标准库与第三方库 ----------
import os  # 路径拼接、文件是否存在
import numpy as np  # 数值数组（此处主要配合 sklearn）
import pandas as pd  # 读 CSV、表格特征

# sklearn：预处理、管道、交叉验证、模型
from sklearn.compose import ColumnTransformer  # 对不同列应用不同变换
from sklearn.impute import SimpleImputer  # 缺失值填充
from sklearn.model_selection import cross_val_score, StratifiedKFold  # 交叉验证与分层折
from sklearn.pipeline import Pipeline  # 把「多步变换」串成一条流水线
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # 类别独热编码、数值标准化
from sklearn.ensemble import GradientBoostingClassifier  # 梯度提升树分类器

# ---------- 数据路径（可按你的实际存放位置修改）----------
# DATA_DIR：两个 CSV 所在文件夹；与 notebook 同目录时常用 "." 表示当前工作目录。
DATA_DIR = "."
# os.path.join：用系统正确的分隔符拼接路径，避免手写 "/"。
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

# 启动时检查文件是否存在，避免后面 read_csv 才报错难以定位。
# (TRAIN_PATH, TEST_PATH) 为元组，for p in ... 逐个检查。
for p in (TRAIN_PATH, TEST_PATH):
    if not os.path.isfile(p):
        # f"..." 为 f-string，可在字符串内嵌入 {表达式}；\n 为换行。
        raise FileNotFoundError(
            f"未找到文件: {os.path.abspath(p)}\n"
            "请从 Kaggle Titanic 页面下载 train.csv 与 test.csv，放入 DATA_DIR 后重试。"
        )

print("数据目录:", os.path.abspath(DATA_DIR))
print("train:", TRAIN_PATH, "| test:", TEST_PATH)

数据目录: /home/featurize/work/zzy/self_learning/pytorch_learning
train: ./train.csv | test: ./test.csv


In [4]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """在原始 Titanic 表上增加衍生特征；训练集与测试集调用同一函数，保证处理一致。

    类型标注说明（typing）：
        df: pd.DataFrame  — 参数应为 pandas 二维表；
        -> pd.DataFrame    — 返回值也是 DataFrame（仅作文档提示，运行时不强制校验）。
    """
    # copy()：复制一份数据，避免在原 df 上直接改列名/加列，影响调用方手里的表。
    out = df.copy()

    # str.extract(正则)：对 Name 列逐行做正则匹配，取出「第一个捕获组」作为 Title。
    # r"..." 为原始字符串，反斜杠按字面处理，方便写正则。
    # 模式含义：空格 + 一段字母 + 英文句点，对应 " Mr." / " Mrs." 中的称谓部分。
    # expand=False：返回与行对齐的 Series（单列）；若为多捕获组可用 expand=True 得到多列。
    out["Title"] = out["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

    # rare 为 set（集合）：语义是「这些称谓样本少，合并为一类」，合并后统计更稳。
    rare = {"Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"}
    # Series.replace(A, B)：若 A 为集合/列表，则集合中出现的旧值一律替换为 B（此处为 "Rare"）。
    out["Title"] = out["Title"].replace(rare, "Rare")

    # replace({旧: 新, ...})：用字典做一一映射，把法语等拼写统一成英语常见称谓。
    out["Title"] = out["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})

    # fillna("Unknown")：正则未匹配到的姓名会得到 NaN，这里统一标成 "Unknown"。
    out["Title"] = out["Title"].fillna("Unknown")

    # 向量化加法：SibSp（兄弟姐妹/配偶数）+ Parch（父母/子女数）+ 本人 1 = 家庭总人数。
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1

    # 先得到布尔 Series（FamilySize == 1），再 astype(int) 转成 0/1，便于后续当作数值特征。
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)

    # Cabin 缺失较多：先填 "U"（Unknown），再转 str，取首字母作为甲板 Deck 的粗粒度类别。
    out["Deck"] = out["Cabin"].fillna("U").astype(str).str[0]
    return out


def build_xy(train: pd.DataFrame):
    """从训练集构造 sklearn 常用的 X（特征矩阵）与 y（一维标签数组）。

    返回值：
        X — 只含模型输入特征，已去掉 ID、原始文本列和标签列；
        y — shape 为 (n_samples,) 的 numpy 数组，元素为 0/1（是否幸存）。
    """
    t = add_features(train)

    # .values：把 Survived 这一列从 pandas Series 转为 numpy.ndarray，与 sklearn 接口习惯一致。
    y = t["Survived"].values

    # 需要从特征里剔除的列：标签、主键、以及 Ticket/Cabin/Name 等原始高基数或已提炼过的列。
    drop_cols = {"Survived", "PassengerId", "Name", "Ticket", "Cabin"}

    # 列表推导式 [c for c in drop_cols if c in t.columns]：
    # 只删除「当前表 t 里确实存在」的列名，避免测试集若少某列时报错（此处 train 必有 Survived）。
    # drop(columns=...)：按列名删除，返回新表赋给 X。
    X = t.drop(columns=[c for c in drop_cols if c in t.columns])
    # Python 可返回多个值，实际打包成一个元组 (X, y)，调用处写作 X, y = build_xy(train_df)。
    return X, y


def build_x_test(test: pd.DataFrame) -> pd.DataFrame:
    """从测试集构造与训练集列含义一致的 X_test（无 Survived，不能泄露标签）。"""
    t = add_features(test)

    # 测试集没有 Survived，因此 drop_cols 不含它；但要同样去掉 PassengerId 与原始文本列。
    drop_cols = {"PassengerId", "Name", "Ticket", "Cabin"}
    return t.drop(columns=[c for c in drop_cols if c in t.columns])

In [5]:
# ---------- 1. 读入原始表 ----------
# read_csv：默认第一行为列名；得到 pandas DataFrame。
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
# .shape：(行数, 列数)，用于快速确认读入是否成功。
print("train 形状:", train_df.shape, "| test 形状:", test_df.shape)

# ---------- 2. 构造训练用 X（特征）、y（标签）----------
# build_xy 内部会调用 add_features，并去掉 Survived / 文本列等。
X, y = build_xy(train_df)

# 列名分为两类，后面 ColumnTransformer 会分别处理（数值 vs 类别）。
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"]
categorical_features = ["Sex", "Embarked", "Title", "Deck"]

# ---------- 3. 数值列流水线：缺失值 → 标准化 ----------
# Pipeline(steps=[(名字, 估计器), ...])：按顺序执行；名字用于网格搜索时访问某一步。
# SimpleImputer(strategy="median")：用列中位数填 NaN（对异常值比均值稳健）。
# StandardScaler：零均值单位方差，让不同量纲的数值可比，利于树以外的模型；树模型对单调变换不敏感但统一写也无妨。
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# ---------- 4. 类别列流水线：缺失值 → 独热编码 ----------
# most_frequent：缺失填该列出现最多的类别。
# OneHotEncoder(handle_unknown="ignore")：若测试集出现训练未见过的类别，对应列全 0，不报错。
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# ---------- 5. 列变换器：对指定列名分别套用上面的两条子流水线 ----------
# remainder="drop" 为默认：未列在 num/cat 中的列会丢弃（此处已显式列出全部使用列）。
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# ---------- 6. 分类器超参数（可调整以试 Leaderboard）----------
model = GradientBoostingClassifier(
    n_estimators=200,  # 树棵数，越大越复杂、更易过拟合
    max_depth=3,  # 每棵树的深度
    learning_rate=0.05,  # 学习率，常与 n_estimators 权衡
    min_samples_split=5,  # 分裂内部节点所需最小样本数
    random_state=42,  # 随机种子，便于结果可复现
)

# ---------- 7. 总流水线：先 preprocess，再 model ----------
# 之后 fit/predict 都只调用 clf 即可，避免「只 transform 训练集却忘了 test」的错误。
clf = Pipeline(steps=[("prep", preprocess), ("clf", model)])

# ---------- 8. 分层 K 折交叉验证（只用训练数据 X, y）----------
# StratifiedKFold：每一折里 0/1 比例接近全数据，适合不平衡二分类。
# shuffle=True：先打乱再分折，减少顺序偏差。
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# cross_val_score：返回每一折的得分；这里用整段 Pipeline clf，内部每折会 fit。
# scoring="accuracy"：准确率 = 预测正确的比例。
scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")
print(f"5-fold CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

# ---------- 9. 在完整训练集上重新训练（用于最终提交）----------
# 交叉验证只用于估计泛化表现；最终模型要用全部 train 信息。
clf.fit(X, y)
print("训练完成。")

train 形状: (891, 12) | test 形状: (418, 11)
5-fold CV accuracy: 0.8451 (+/- 0.0178)
训练完成。


In [6]:
# ---------- 1. 测试集特征（与训练时同一套 add_features / 列剔除逻辑）----------
# 注意：这里不能用 Survived；build_x_test 只保留与 X 同结构的列。
X_test = build_x_test(test_df)

# ---------- 2. 提交文件需要的乘客 ID（与 Kaggle 样例格式一致）----------
# 来自原始 test_df，与 X_test 行顺序一一对应（read_csv 默认顺序未打乱）。
ids = test_df["PassengerId"]

# ---------- 3. 预测 ----------
# clf.predict：Pipeline 先对 X_test 做与训练相同的 prep，再输出类别 0/1。
# 梯度提升等 sklearn 分类器默认 predict 为类别标签，不是概率。
preds = clf.predict(X_test)

# ---------- 4. 拼成提交表并写出 CSV ----------
# DataFrame({列名: Series, ...})：按列构造表；两列长度须相同。
# Survived 需为整数 0/1，astype(int) 与竞赛要求一致。
submission = pd.DataFrame({"PassengerId": ids, "Survived": preds.astype(int)})
out_path = os.path.join(DATA_DIR, "submission.csv")
# index=False：不写出行索引 0,1,2...（Kaggle 只要求两列）。
submission.to_csv(out_path, index=False)
print(f"已保存: {os.path.abspath(out_path)}")
# head(10)：查看前 10 行，确认格式无误。
submission.head(10)

已保存: /home/featurize/work/zzy/self_learning/pytorch_learning/submission.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


## 可选：换模型试 Public Leaderboard

在训练单元格中把 `GradientBoostingClassifier(...)` 换成例如：

```python
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=2, random_state=42
)
```

重新跑交叉验证与预测单元格，对比线上分数。